<a href="https://colab.research.google.com/github/Akhilesh-Ankur09/multimodal-genai-video-pipeline/blob/main/notebooks/genai_video_generation_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Setup & Imports
import os, random, json, requests, asyncio, base64, time
from io import BytesIO
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import moviepy.config as mp_config


# Force-disable ImageMagick to prevent errors
mp_config.IMAGEMAGICK_BINARY = ""
from moviepy.editor import *

# Install missing specific libraries
!pip install edge-tts whisper-timestamped

import edge_tts
import whisper_timestamped as whisper
from google.colab import drive
import os

# Using force_remount=True solves the "Mountpoint already contains files" error
drive.mount('/content/drive', force_remount=True)

# Re-confirming your project paths
BASE_DIR = "/content/drive/MyDrive/Flora_Fauna_Facts"
RAW_VIDEO_DIR = os.path.join(BASE_DIR, "raw_videos")
AUDIO_DIR = os.path.join(BASE_DIR, "audio")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

# This will create the folders for your new run if they were deleted
for d in [RAW_VIDEO_DIR, AUDIO_DIR, OUTPUT_DIR]:
    os.makedirs(d, exist_ok=True)

print("✅ Drive mounted and project folders ready for a new run!")

# Constants
FPS = 30
RESOLUTION = (1080, 1920) # 9:16

print("✅ Environment Ready.")

In [ ]:
!pip install -q -U google-genai

In [ ]:
import os
import time
from google import genai
from google.genai import types
from google.colab import userdata

# --- API KEYS ---
# Load API keys securely from Colab Secrets
# Load Gemini keys
GEMINI_KEYS = [
    userdata.get("GEMINI_KEY_1"),
    userdata.get("GEMINI_KEY_2"),
    userdata.get("GEMINI_KEY_3"),
    userdata.get("GEMINI_KEY_4")
]

# Load Pexels key
PEXELS_API_KEY = userdata.get("PEXELS_API_KEY")

# Store in environment variable for compatibility
os.environ["PEXELS_API_KEY"] = PEXELS_API_KEY

# --- ROLLBACK STATE ---
current_key_index = 0

def get_current_client():
    """Returns a Gemini client using the currently active key."""
    return genai.Client(
        api_key=GEMINI_KEYS[current_key_index],
        http_options=types.HttpOptions(api_version="v1")
    )

# Initial client setup
client = get_current_client()

def rotate_key():
    """Surgically switches to the next project key."""
    global current_key_index, client
    if current_key_index < len(GEMINI_KEYS) - 1:
        current_key_index += 1
        print(f"🔄 Rolling back to Project {current_key_index + 1}...")
        client = get_current_client() # Re-initializes 'client' with the new key
        return True
    else:
        print("❌ ALL PROJECTS EXHAUSTED for today.")
        return False

def call_gemini(prompt):
    """Acts as the 'Brain' for scripts and queries."""
    model_id = "gemini-2.5-flash"

    while current_key_index < len(GEMINI_KEYS):
        try:
            response = client.models.generate_content(
                model=model_id,
                contents=prompt,
                config=types.GenerateContentConfig(temperature=0.7, max_output_tokens=1000)
            )
            print(f"✅ Project {current_key_index + 1} used.")
            return response.text.strip()

        except Exception as e:
            error_msg = str(e).upper()
            if "429" in error_msg or "RESOURCE_EXHAUSTED" in error_msg:
                if not rotate_key():
                    return "ERROR: ALL QUOTAS EXHAUSTED"
                continue # Retry with new key
            else:
                print(f"❌ Gemini Error: {str(e)}")
                return "ERROR."

print(f"✅ API Rollback Configuration complete. rotate_key() is now defined.")

In [ ]:
# Cell 3: Topic & Script Generation (Manual Subject Input)

# --- CHANGE YOUR SUBJECT HERE ---
INPUT_SUBJECT = "Swan"
# --------------------------------

# 1. Prepare Subject and determine category automatically
SUBJECT = INPUT_SUBJECT.strip()
category_prompt = f"Is '{SUBJECT}' fauna or flora? Return only the word 'fauna' or 'flora'."
CATEGORY = call_gemini(category_prompt).strip().lower()

# 2. Generate Script
# Focuses on interesting facts while staying under the word limit for a Short
script_prompt = (f"Write a 4-sentence interesting fact about the {SUBJECT}. "
                 f"Focus on unique physical traits or behaviors. Keep it under 50 words total.")
SCRIPT = call_gemini(script_prompt)

# 3. Generate High-Accuracy Search Terms
# --- Updated Search Term Logic ---
search_prompt = (
    f"Provide 3 Pexels search terms for the {SUBJECT}. "
    f"CRITICAL: Every search term MUST include the words '{SUBJECT}'. "
    f"Focus on the behavior mentioned: {SCRIPT[:50]}... "
    f"Format: {SUBJECT} action1, {SUBJECT} action2, {SUBJECT} habitat"
)
raw_queries = call_gemini(search_prompt).split(",")
PEXELS_QUERIES = [t.strip() for t in raw_queries]

print(f"🎯 Subject Locked: {SUBJECT} ({CATEGORY})")
print(f"📜 Script: {SCRIPT}")
print(f"🔍 Pexels Queries: {PEXELS_QUERIES}")

In [ ]:
# Cell 4: Audio & Timestamps
AUDIO_PATH = os.path.join(AUDIO_DIR, f"{SUBJECT}_voice.mp3")

async def make_audio():
    communicate = edge_tts.Communicate(SCRIPT, "en-US-AndrewNeural")
    await communicate.save(AUDIO_PATH)

await make_audio()
AUDIO_CLIP = AudioFileClip(AUDIO_PATH)

# Word-level sync using Whisper
model = whisper.load_model("tiny", device="cpu")
result = whisper.transcribe(model, whisper.load_audio(AUDIO_PATH), language="en")

WORD_TIMESTAMPS = []
for segment in result['segments']:
    for word in segment['words']:
        WORD_TIMESTAMPS.append({"text": word['text'].upper(), "start": word['start'], "end": word['end']})

print(f"✅ Audio synced. Total duration: {AUDIO_CLIP.duration}s")

In [ ]:
import requests
import random
import time
from io import BytesIO
from PIL import Image
import numpy as np
import os
from moviepy.editor import VideoFileClip

def frame_matches_subject(frame, subject_name):
    """Surgically updated: No internal catch, so rollback can see the error."""
    if frame is None or np.all(frame == 0):
        return False

    # Convert numpy frame directly to PIL Image
    pil_img = Image.fromarray(frame).convert("RGB")

    prompt = (
        f"Does this image clearly show a {subject_name} in a natural environment? "
        f"Answer ONLY YES or NO. "
        f"Reject (NO) if it shows humans, factories, cooking, or generic landscapes without the {subject_name}."
    )

    # We removed the try/except here so the 429 error bubbles up to the loop below
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=[prompt, pil_img]
    )

    answer = response.text.strip().upper()
    return "YES" in answer

def download_and_smart_crop(url, idx):
    """Downloads and crops to 9:16 without distortion."""
    path = os.path.join(RAW_VIDEO_DIR, f"temp_{idx}.mp4")
    r = requests.get(url)
    with open(path, "wb") as f: f.write(r.content)

    try:
        clip = VideoFileClip(path)
        clip_scaled = clip.resize(width=1080)

        # Smart Crop logic
        if clip_scaled.h > 1920:
            final_clip = clip_scaled.crop(y_center=clip_scaled.h/2, height=1920)
        else:
            final_clip = clip_scaled.on_color(size=(1080, 1920), color=(0,0,0), pos='center')

        return final_clip.set_fps(FPS)
    except Exception as e:
        print(f"Error processing clip {idx}: {e}")
        return None

# --- EXECUTION ---
FINAL_VALIDATED_CLIPS = []
headers = {"Authorization": os.environ.get("PEXELS_API_KEY")}

print(f"🔎 Starting search for: {SUBJECT}...")

for q in PEXELS_QUERIES:
    if len(FINAL_VALIDATED_CLIPS) >= 5: break

    # Higher per_page to give Gemini more candidates to choose from
    random_page = random.randint(1, 5)
    api_url = f"https://api.pexels.com/videos/search?query={q}&per_page=20&page={random_page}"

    try:
        response = requests.get(api_url, headers=headers).json()
        vids = response.get('videos', [])
        random.shuffle(vids)

        for v in vids:
            if len(FINAL_VALIDATED_CLIPS) >= 5: break

            video_url = v['video_files'][0]['link']
            temp_clip = download_and_smart_crop(video_url, len(FINAL_VALIDATED_CLIPS))
            if temp_clip is None: continue

            # Validate with Vision (Check middle frame)
            test_frame = temp_clip.get_frame(temp_clip.duration / 2)

            # --- THE ROLLBACK ENGINE ---
            verified = False
            try:
                if frame_matches_subject(test_frame, SUBJECT):
                    verified = True
                else:
                    print(f"   ❌ Rejected clip: No {SUBJECT} detected.")
                    temp_clip.close()

            except Exception as vision_err:
                # Catch the quota exhaustion error
                if "429" in str(vision_err) or "RESOURCE_EXHAUSTED" in str(vision_err):
                    print("⚠️ Project exhausted. Attempting Rollback...")
                    if rotate_key(): # Function from your configuration cell
                        # RETRY the same frame with the new key immediately
                        try:
                            if frame_matches_subject(test_frame, SUBJECT):
                                verified = True
                            else:
                                print(f"   ❌ Rejected by fresh project.")
                                temp_clip.close()
                        except:
                            temp_clip.close()
                    else:
                        print("🛑 All project quotas exhausted. Stopping search.")
                        break
                else:
                    print(f"⚠️ Vision Error: {vision_err}")
                    temp_clip.close()

            if verified:
                print(f"   ✅ Verified {SUBJECT} found! Total: {len(FINAL_VALIDATED_CLIPS)+1}/5")
                FINAL_VALIDATED_CLIPS.append(temp_clip)

            # Important: 4-second sleep to stay under the Per-Minute limit (RPM)
            time.sleep(4)

    except Exception as e:
        print(f"Search error for query '{q}': {e}")

print(f"🏁 Done! Found {len(FINAL_VALIDATED_CLIPS)} genuine, verified clips.")

In [ ]:
# Cell 6: Assembly (Optimized for Subtitle Visibility and Safe Zones)
def create_caption(text):
    # Create transparent canvas matching the 1080x1920 resolution
    img = Image.new('RGBA', RESOLUTION, (0, 0, 0, 0))
    draw = ImageDraw.Draw(img)

    try:
        # Size 130 for high impact on mobile screens
        font = ImageFont.truetype("/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 130)
    except:
        font = ImageFont.load_default()

    # POSITIONING: Moved from 1500 to 1400.
    # This prevents the caption from being covered by the YouTube 'Channel Name' and 'Description'
    # Anchor "mm" keeps the word perfectly centered horizontally.
    draw.text((540, 1400), text, font=font, fill="yellow",
              anchor="mm", stroke_width=10, stroke_fill="black") # Increased stroke for even better contrast

    return ImageClip(np.array(img))

# 1. Background Sequence
# We use 'compose' method to ensure that different clip metadata doesn't break the render
print("🎞️ Stitching background clips...")
per_clip_duration = AUDIO_CLIP.duration / len(FINAL_VALIDATED_CLIPS)
bg_vids = [v.subclip(0, min(per_clip_duration, v.duration)) for v in FINAL_VALIDATED_CLIPS]
BASE_VIDEO = concatenate_videoclips(bg_vids, method="compose")

# 2. Synchronized Captions
print("💬 Overlaying synchronized captions...")
caps = []
for w in WORD_TIMESTAMPS:
    duration = max(0.2, w['end'] - w['start']) # Minimum 0.2s so words don't flicker too fast
    caption_clip = (create_caption(w['text'])
                    .set_start(w['start'])
                    .set_duration(duration)
                    .set_position(('center', 'center'))) # Explicitly center the clip object
    caps.append(caption_clip)

# 3. Final Render
# The order [BASE_VIDEO] + caps is CRITICAL. Base must be index 0.
FINAL_VIDEO = CompositeVideoClip([BASE_VIDEO] + caps).set_audio(AUDIO_CLIP)

# Clean filename logic
clean_subject = SUBJECT.replace(' ', '_').replace('/', '_')
FINAL_PATH = os.path.join(OUTPUT_DIR, f"{clean_subject}_Final_Short.mp4")

print(f"🎬 Rendering Final Video to Drive: {FINAL_PATH}...")

# Added temp_audiofile and remove_temp to prevent write errors in Colab environments
FINAL_VIDEO.write_videofile(
    FINAL_PATH,
    fps=FPS,
    codec="libx264",
    audio_codec="aac",
    temp_audiofile='temp-audio.m4a',
    remove_temp=True,
    threads=4 # Speed up rendering using multiple cores
)

print(f"✅ SUCCESS! Your video is ready at: {FINAL_PATH}")

In [ ]:
from IPython.display import HTML
from base64 import b64encode
import os

# 1. Verify the file exists before trying to open it
if os.path.exists(FINAL_PATH):
    # 2. Read the video file and encode it
    mp4 = open(FINAL_PATH, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4).decode()

    # 3. Display using HTML5 video player
    display(HTML(f"""
    <div style="display: flex; justify-content: center;">
        <video width="320" height="560" controls autoplay muted loop>
            <source src="{data_url}" type="video/mp4">
            Your browser does not support the video tag.
        </video>
    </div>
    """))
else:
    print(f"❌ Video not found at: {FINAL_PATH}")